## Setup: Install dependencies and clone repo

In [ ]:
# Install packages
!pip install -q torch datasets transformers numpy matplotlib
print("Dependencies installed")

In [ ]:
# Clone repo
!git clone https://github.com/ahuliangbo/NLP-FINAL.git
%cd NLP-FINAL/src
print("Repository cloned")

## Step 1: Download WikiText-2 Dataset

In [ ]:
# Download WikiText-2
!python download_wikitext2.py --output data/wikitext2_5k.txt --max-lines 5000
print("WikiText-2 downloaded")

## Step 2: Import and Setup Logging

In [ ]:
import torch
import json
import time
import psutil
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Dict, List
import numpy as np

from models.decoder_only import DecoderConfig, MiniDecoder, build_few_shot_prompt
from utils import SimpleTokenizer, set_seed
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Experiment class
class ExperimentTracker:
    """Track metrics during training."""
    def __init__(self, name: str):
        self.name = name
        self.losses = []
        self.accuracies = []
        self.epoch_times = []
        self.start_time = None
        self.total_time = 0
        self.peak_memory = 0
        
    def start(self):
        self.start_time = time.time()
        
    def end_epoch(self, loss: float, accuracy: float):
        elapsed = time.time() - self.start_time
        self.losses.append(loss)
        self.accuracies.append(accuracy)
        self.epoch_times.append(elapsed)
        self.total_time += elapsed
        
        # Track memory
        if torch.cuda.is_available():
            mem = torch.cuda.memory_allocated() / 1024**3
            self.peak_memory = max(self.peak_memory, mem)
        
    def summary(self) -> Dict:
        return {
            "name": self.name,
            "total_time_sec": self.total_time,
            "avg_epoch_time_sec": np.mean(self.epoch_times) if self.epoch_times else 0,
            "peak_memory_gb": self.peak_memory,
            "final_loss": self.losses[-1] if self.losses else None,
            "final_accuracy": self.accuracies[-1] if self.accuracies else None,
            "best_loss": min(self.losses) if self.losses else None,
        }

print("Experiment tracking setup done")

## Step 3: Data Loading Functions

In [ ]:
class PatternDataset(Dataset):
    """Dataset made from sequences."""
    def __init__(self, sequences: List[str], tokenizer: SimpleTokenizer, max_length: int = None):
        self.sequences = [seq for seq in sequences if seq.strip()]
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.sequences)

    def __getitem__(self, index: int):
        tokens = self.tokenizer.encode(self.sequences[index], add_special_tokens=True)
        if self.max_length and len(tokens) > self.max_length:
            tokens = tokens[:self.max_length]
        return {"input_ids": tokens}

def collate_fn(batch, tokenizer: SimpleTokenizer):
    input_ids = [item["input_ids"] for item in batch]
    padded, attention = tokenizer.pad(input_ids)
    return {
        "input_ids": padded,
        "attention_mask": attention,
        "labels": padded.clone(),
    }

def load_sequences(path: Path, max_lines: int = None) -> List[str]:
    sequences = []
    with open(path, "r", encoding="utf-8") as handle:
        for i, line in enumerate(handle):
            if max_lines and i >= max_lines:
                break
            stripped = line.strip()
            if stripped:
                sequences.append(stripped)
    return sequences

print("Data loading functions ready")

## Step 4: Training Function

In [ ]:
def train_experiment(config: DecoderConfig, data_path: Path, optimizer_type: str, 
                     epochs: int = 10, batch_size: int = 16, lr: float = 3e-4) -> ExperimentTracker:
    """Train model with specified configuration."""
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tracker = ExperimentTracker(f"{config.embed_dim}D-{config.num_layers}L-{optimizer_type}")
    
    # Load data
    sequences = load_sequences(data_path)
    tokenizer = SimpleTokenizer(sequences)
    dataset = PatternDataset(sequences, tokenizer, max_length=config.max_seq_len)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, 
                        collate_fn=lambda b: collate_fn(b, tokenizer))
    
    # Update config
    config.vocab_size = tokenizer.vocab_size
    config.pad_token_id = tokenizer.token_id(tokenizer.pad_token)
    config.bos_token_id = tokenizer.token_id(tokenizer.bos_token)
    config.eos_token_id = tokenizer.token_id(tokenizer.eos_token)
    
    # Create model
    model = MiniDecoder(config).to(device)
    
    # Create optimizer
    if optimizer_type == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    elif optimizer_type == "adamw":
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    elif optimizer_type == "sgd":
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    
    # Training loop
    print(f"\n{'='*60}")
    print(f"Training: {tracker.name}")
    print(f"Device: {device} | Params: {sum(p.numel() for p in model.parameters()):,}")
    print(f"{'='*60}")
    
    for epoch in range(1, epochs + 1):
        tracker.start()
        model.train()
        epoch_loss = 0.0
        epoch_acc = 0.0
        steps = 0
        
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model.forward_train(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs["loss"]

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            epoch_loss += float(loss.item())
            epoch_acc += float(outputs["accuracy"].item())
            steps += 1
        
        avg_loss = epoch_loss / max(1, steps)
        avg_acc = epoch_acc / max(1, steps)
        tracker.end_epoch(avg_loss, avg_acc)
        
        print(f"Epoch {epoch:02d}/{epochs} | Loss: {avg_loss:.4f} | Accuracy: {avg_acc:.4f} | Time: {tracker.epoch_times[-1]:.2f}s")
    
    return tracker

print("Training function ready")

## Step 5: Run Comparative Experiments

### Experiment 1: Model Size Comparison
**Research Question:** How does model size affect training efficiency and final loss?

In [ ]:
set_seed(42)
results_size = {}

# Small model (2 layers, 64D)
config_small = DecoderConfig(
    vocab_size=1000,  # will be updated
    embed_dim=64,
    num_heads=2,
    ff_dim=128,
    num_layers=2,
    dropout=0.1,
    max_seq_len=128,
)
tracker_small = train_experiment(config_small, Path("data/wikitext2_5k.txt"), "adam", epochs=5, batch_size=16)
results_size["small"] = tracker_small

In [ ]:
# Medium model (4 layers, 128D)
config_medium = DecoderConfig(
    vocab_size=1000,
    embed_dim=128,
    num_heads=4,
    ff_dim=256,
    num_layers=4,
    dropout=0.1,
    max_seq_len=128,
)
tracker_medium = train_experiment(config_medium, Path("data/wikitext2_5k.txt"), "adam", epochs=5, batch_size=16)
results_size["medium"] = tracker_medium

In [ ]:
# Large model (6 layers, 256D)
config_large = DecoderConfig(
    vocab_size=1000,
    embed_dim=256,
    num_heads=8,
    ff_dim=512,
    num_layers=6,
    dropout=0.1,
    max_seq_len=128,
)
tracker_large = train_experiment(config_large, Path("data/wikitext2_5k.txt"), "adam", epochs=5, batch_size=16)
results_size["large"] = tracker_large

### Experiment 2: Optimizer Comparison
**Research Question:** Which optimizer converges fastest and achieves lowest loss?

In [ ]:
set_seed(42)
results_optimizer = {}

config_base = DecoderConfig(
    vocab_size=1000,
    embed_dim=128,
    num_heads=4,
    ff_dim=256,
    num_layers=4,
    dropout=0.1,
    max_seq_len=128,
)

for opt_type in ["adam", "adamw", "sgd"]:
    tracker = train_experiment(config_base, Path("data/wikitext2_5k.txt"), opt_type, epochs=5, batch_size=16, lr=3e-4)
    results_optimizer[opt_type] = tracker

## Step 6: Visualize Results

In [ ]:
# Plot 1: Model Size - Loss Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for name, tracker in results_size.items():
    axes[0].plot(tracker.losses, marker='o', label=name)
axes[0].set_xlabel("Epoch", fontsize=12)
axes[0].set_ylabel("Loss", fontsize=12)
axes[0].set_title("Model Size Comparison: Loss Curves", fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot accuracy
for name, tracker in results_size.items():
    axes[1].plot(tracker.accuracies, marker='o', label=name)
axes[1].set_xlabel("Epoch", fontsize=12)
axes[1].set_ylabel("Token Accuracy", fontsize=12)
axes[1].set_title("Model Size Comparison: Accuracy", fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("exp1_model_size.png", dpi=100, bbox_inches='tight')
plt.show()

print("Model size comparison plot saved")

In [ ]:
# Plot 2: Optimizer Comparison - Loss Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for opt_type, tracker in results_optimizer.items():
    axes[0].plot(tracker.losses, marker='s', label=opt_type)
axes[0].set_xlabel("Epoch", fontsize=12)
axes[0].set_ylabel("Loss", fontsize=12)
axes[0].set_title("Optimizer Comparison: Loss Curves", fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot timing comparison
opt_types = list(results_optimizer.keys())
total_times = [results_optimizer[opt].total_time for opt in opt_types]
axes[1].bar(opt_types, total_times, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
axes[1].set_ylabel("Total Training Time (seconds)", fontsize=12)
axes[1].set_title("Optimizer Comparison: Training Time", fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig("exp2_optimizer.png", dpi=100, bbox_inches='tight')
plt.show()

print("Optimizer comparison plot saved")

## Step 7: Summary Statistics

In [ ]:
import pandas as pd

# Experiment 1: Model Size
print("\n" + "="*80)
print("EXPERIMENT 1: MODEL SIZE COMPARISON")
print("="*80)

exp1_data = []
for name, tracker in results_size.items():
    summary = tracker.summary()
    exp1_data.append({
        "Model": name,
        "Total Time (s)": f"{summary['total_time_sec']:.1f}",
        "Avg Epoch (s)": f"{summary['avg_epoch_time_sec']:.1f}",
        "Final Loss": f"{summary['final_loss']:.4f}",
        "Best Loss": f"{summary['best_loss']:.4f}",
        "Final Acc": f"{summary['final_accuracy']:.4f}",
        "Peak Memory (GB)": f"{summary['peak_memory_gb']:.2f}",
    })

df1 = pd.DataFrame(exp1_data)
print(df1.to_string(index=False))

print("\n" + "="*80)
print("EXPERIMENT 2: OPTIMIZER COMPARISON")
print("="*80)

exp2_data = []
for opt_type, tracker in results_optimizer.items():
    summary = tracker.summary()
    exp2_data.append({
        "Optimizer": opt_type.upper(),
        "Total Time (s)": f"{summary['total_time_sec']:.1f}",
        "Avg Epoch (s)": f"{summary['avg_epoch_time_sec']:.1f}",
        "Final Loss": f"{summary['final_loss']:.4f}",
        "Best Loss": f"{summary['best_loss']:.4f}",
        "Final Acc": f"{summary['final_accuracy']:.4f}",
    })

df2 = pd.DataFrame(exp2_data)
print(df2.to_string(index=False))

## Step 8: Key Findings & Analysis

In [ ]:

# Find best in each category
best_loss_model = min(results_size.items(), key=lambda x: min(x[1].losses))
best_time_model = min(results_size.items(), key=lambda x: x[1].total_time)

best_opt = min(results_optimizer.items(), key=lambda x: min(x[1].losses))
fastest_opt = min(results_optimizer.items(), key=lambda x: x[1].total_time)

print(f"""
MODEL SIZE FINDINGS:
  • Best final loss: {best_loss_model[0]} model ({min(best_loss_model[1].losses):.4f})
  • Fastest training: {best_time_model[0]} model ({best_time_model[1].total_time:.1f}s total)


OPTIMIZER FINDINGS:
  • Best final loss: {best_opt[0].upper()} optimizer ({min(best_opt[1].losses):.4f})
  • Fastest training: {fastest_opt[0].upper()} optimizer ({fastest_opt[1].total_time:.1f}s total)


""")


## Step 9: Save Results and Checkpoint

In [ ]:
# Save experiment results as JSON
results_summary = {
    "experiment_1_model_size": {k: v.summary() for k, v in results_size.items()},
    "experiment_2_optimizer": {k: v.summary() for k, v in results_optimizer.items()},
}

with open("experiment_results.json", "w") as f:
    json.dump(results_summary, f, indent=2)

print("Results saved to experiment_results.json")

## Additional Experiment: Positional Encoding Ablation
This runs `src/positional_ablation.py` which trains sinusoidal, learned, and RoPE models, then saves plots to `src/outputs/`.

In [ ]:
# Ensure required packages (datasets may be needed by the script)
!pip install -q datasets matplotlib

# Run the positional ablation script (will save images under src/outputs)
!python positional_ablation.py || true

# Display generated images and results if present
import os
from IPython.display import Image, display
out_dir = 'outputs'
imgs = ['pos_ablation_loss.png', 'pos_ablation_perplexity.png']
for im in imgs:
    p = os.path.join(out_dir, im)
    if os.path.exists(p):
        display(Image(p))
    else:
        print(f'Missing: {p}')

# Print JSON summary if available
j = os.path.join(out_dir, 'pos_ablation_results.json')
if os.path.exists(j):
    import json
    print('Positional ablation results:')
    print(json.dumps(json.load(open(j)), indent=2))
else:
    print('No results JSON found at', j)